# CNN log-mel + augmentasi
        **audio › classification** · notebook `logmel_cnn_aug`

        **Kapan dipakai:** Sama dengan di atas dengan geser waktu, derau, dan gain acak untuk ketahanan.

        **Jebakan umum:** Pada data sangat kecil (1200 klip) augmentasi memperlambat konvergensi: di demo ini belum mengungguli tanpa augmentasi bahkan setelah 120 epoch. Ukur sendiri; manfaatnya muncul pada data lebih banyak/kondisi uji berderau. Augmentasi harus mencerminkan kondisi nyata.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** Speech Commands mini (8 kata: down/go/left/no/right/stop/up/yes; 1200 latih / 320 uji; 1 detik, 16 kHz). Bobot pretrained dari `pretrained/` (offline); GPU otomatis bila ada.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Format data: folder WAV + CSV `path,label` (path relatif ke `DATA_ROOT`). Klip dipotong/di-pad ke `LENGTH` sampel (16 kHz).

In [ ]:
DATA_ROOT = ROOT / "data_latihan/audio/classification/speech_commands_mini"
TRAIN_CSV, TEST_CSV = DATA_ROOT / "train.csv", DATA_ROOT / "test.csv"
TEST_LABELS = DATA_ROOT / "test_labels.csv"        # kunci jawaban test (hanya latihan); None di lomba
LENGTH = 16000                                        # 1 detik
SEED, VALID_SIZE = 42, 0.2
MODEL_KEY = "logmel_cnn_aug"
OUT = out_dir("audio", "classification", MODEL_KEY); set_seed(SEED)
EPOCHS, AUG, LR, BATCH = 120, True, 3e-3, 32

## 2. Baca audio

In [ ]:
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from superdb.audio import load_wavs, plot_wave_mel
tr, te = pd.read_csv(TRAIN_CSV), pd.read_csv(TEST_CSV)
le = LabelEncoder(); y = le.fit_transform(tr["label"]); CLASSES = list(le.classes_)
X = load_wavs(tr["path"], DATA_ROOT, LENGTH); Xte = load_wavs(te["path"], DATA_ROOT, LENGTH)
print(X.shape, Xte.shape, "| kelas:", CLASSES, "| durasi:", LENGTH / 16000, "dtk")
plot_wave_mel(X[0], f"contoh '{tr['label'][0]}'"); plt.show()
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=VALID_SIZE, random_state=SEED, stratify=y)

## 3. Model: mel-spektrogram dihitung di GPU di dalam jaringan
Masukan model = gelombang mentah (B, T); lapisan pertama mengubahnya jadi log-mel; lalu CNN 2D kecil.

In [ ]:
import torch
from superdb.audio import make_logmel_cnn, train_audio_cnn, predict_proba_cnn, device
net = make_logmel_cnn(len(CLASSES)); print("device:", device(), "|", round(sum(p.numel() for p in net.parameters()) / 1e3), "ribu parameter")

## 4. Latih
Augmentasi (bila aktif): geser waktu ±0.1 dtk, derau SNR 15-30 dB, gain 0.7-1.3.

In [ ]:
hist = train_audio_cnn(net, Xtr, ytr, Xva, yva, epochs=EPOCHS, lr=LR, batch_size=BATCH, aug=AUG, seed=SEED)
h = pd.DataFrame(hist); display(h.tail(5).round(4))
fig, ax = plt.subplots(1, 2, figsize=(9, 3)); h.train_loss.plot(ax=ax[0], title="loss latih"); h.valid_acc.plot(ax=ax[1], title="akurasi validation"); plt.tight_layout(); plt.savefig(OUT / "curve.png", dpi=130); plt.show()

## 5. Evaluasi

In [ ]:
proba = predict_proba_cnn(net, Xva); pred = proba.argmax(1)
metrics = classification_metrics(yva, pred, proba, classes=np.arange(len(CLASSES))); display(show_table(metrics))
plot_confusion(yva, pred, labels=np.arange(len(CLASSES)), path=OUT / "confusion.png"); plt.show()

## 6. Prediksi test → submission → simpan

In [ ]:
PRED_TE = predict_proba_cnn(net, Xte).argmax(1)

In [ ]:
sub = pd.DataFrame({"id": te["id"], "label": [CLASSES[i] for i in PRED_TE]}); sub.to_csv(OUT / "submission.csv", index=False); display(sub.head(3))
if TEST_LABELS is not None and Path(TEST_LABELS).exists():
    truth = pd.read_csv(TEST_LABELS); acc = float((truth.set_index("id").loc[sub["id"], "label"].to_numpy() == sub["label"].to_numpy()).mean())
    print(f"akurasi pada test (kunci jawaban): {acc:.4f}"); metrics["test_accuracy"] = acc

In [ ]:
torch.save({"state_dict": net.state_dict(), "classes": CLASSES}, OUT / "model.pt")
net2 = make_logmel_cnn(len(CLASSES)); net2.load_state_dict(torch.load(OUT / "model.pt", map_location="cpu")["state_dict"]); net2.to(device())
agree = float((predict_proba_cnn(net2, Xva[:64]).argmax(1) == pred[:64]).mean()); assert agree > 0.98, f"model muat-ulang beda ({agree:.3f})"
save_json({"valid": metrics, "history": hist}, OUT / "metrics.json"); print(f"OK. hasil di {OUT} (kesepakatan setelah muat ulang {agree:.3f})")

## Langkah berikutnya
- Bandingkan dengan `wav2vec2_*`/`whisper_encoder_*` (transfer learning) pada data yang sama.
- Tambahkan SpecAugment (mask frekuensi/waktu) untuk regularisasi lebih kuat.
- Klip lebih panjang: jendela geser + agregasi.